# Simple GAN — Unconditional Face Generation on CelebA
**Architecture:** DCGAN (Radford et al., 2015)  
**Dataset:** CelebA 64x64 · **Framework:** PyTorch · **Platform:** Google Colab

---
## Purpose
This is the **baseline GAN** before AttGAN. Key differences:

| | Simple GAN (this notebook) | AttGAN |
|---|---|---|
| Conditioning | None — pure random noise | Attribute vector {-1,+1} |
| Output control | None | Edit specific facial attributes |
| Loss | BCE adversarial only | Adv + Classification + Reconstruction |
| Resolution | 64x64 | 128x128 |

---
## Before you run anything

### Step A — Set runtime to GPU
`Runtime → Change runtime type → T4 GPU → Save`

### Step B — Download CelebA from Kaggle (same as AttGAN notebook)
1. Go to https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
2. Download and extract — you need:
   - `img_align_celeba/` folder (unzipped from img_align_celeba.zip — 202,599 .jpg files)
   - `list_attr_celeba.csv`
   - `list_eval_partition.csv`

### Step C — Upload to Google Drive
Create this structure in Drive:
```
My Drive/
  celeba_data/
      img_align_celeba/        <- 202,599 jpg files here
      list_attr_celeba.csv
      list_eval_partition.csv
```
> If you already did this for the AttGAN notebook, you can reuse the same folder — no need to re-upload.

---
## Cell 1 — Clone repo & install

In [ ]:
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL
!pip install -q -r requirements.txt

import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('NO GPU — go to Runtime > Change Runtime Type > T4 GPU')

---
## Cell 2 — Mount Google Drive & verify dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change this if you uploaded to a different path
CELEBA_DRIVE_PATH = '/content/drive/MyDrive/celeba_data'

celeba_dir = Path(CELEBA_DRIVE_PATH)
required = [
    celeba_dir / 'img_align_celeba',
    celeba_dir / 'list_attr_celeba.csv',
    celeba_dir / 'list_eval_partition.csv',
]
all_ok = True
for p in required:
    status = 'OK' if p.exists() else 'MISSING'
    if status == 'MISSING': all_ok = False
    print(f'  [{status}]  {p}')

if not all_ok:
    raise FileNotFoundError('Missing files — see Step C in the header.')

n_imgs = len(list((celeba_dir / 'img_align_celeba').glob('*.jpg')))
print(f'\nImages found: {n_imgs:,}  (expected 202,599)')
print('Dataset ready!')

---
## Cell 3 — Config & device

In [ ]:
import torch
from config import Config

cfg = Config()
cfg.EXPERIMENT_NAME = 'simple_gan'
cfg.CELEBA_DIR      = celeba_dir    # point to Drive dataset
cfg.__init__()                      # creates results/simple_gan/ etc.

# Optional: reduce epochs for a quick smoke test
# cfg.N_EPOCHS = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Experiment   : {cfg.EXPERIMENT_NAME}')
print(f'Device       : {device}')
print(f'Epochs       : {cfg.N_EPOCHS}')
print(f'Results dir  : {cfg.RESULTS_DIR}')

---
## Cell 4 — Load dataset
The train split has 162,770 images. Images are downsampled to 64x64 inside the training loop (DCGAN architecture).

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}   (loaded at 128x128, downsampled to 64x64 in training loop)')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

---
## Cell 5 — Build models
DCGAN architecture:
```
noise z (100x1x1)
   |
   v  Generator: 5x ConvTranspose2d + BatchNorm + ReLU -> Tanh
fake image (3x64x64)
   |
   v  Discriminator: 5x Conv2d + BatchNorm + LeakyReLU -> Sigmoid
real/fake scalar [0, 1]

Loss G: BCE(D(G(z)), 1)
Loss D: BCE(D(x), 1) + BCE(D(G(z)), 0)
```

In [ ]:
from src.simple_gan import build_simple_models

LATENT_DIM = 100
gen, dis = build_simple_models(latent_dim=LATENT_DIM, device=device)

---
## Cell 6 — Train

In [ ]:
from src.simple_gan import train_simple_gan

g_losses, d_losses = train_simple_gan(
    gen, dis, train_loader, cfg, device, latent_dim=LATENT_DIM
)

---
## Cell 7 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

---
## Cell 8 — FID & DACID metrics
Computes **FID** (Frechet Inception Distance) and **DACID** (custom L2 mean feature distance)
on the test split to quantitatively evaluate generation quality.

In [ ]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics_simple_gan
    scores = compute_metrics_simple_gan(gen, test_loader, LATENT_DIM, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')

---
## Cell 9 — Save checkpoint & metrics.json

In [ ]:
import json, torch

# Save model weights
ckpt_path = cfg.CHECKPOINT_DIR / 'simple_gan_final.pt'
torch.save({'gen': gen.state_dict(), 'dis': dis.state_dict()}, ckpt_path)
print(f'Checkpoint -> {ckpt_path}')

# Save metrics for export_results.py
payload = {
    'experiment': cfg.EXPERIMENT_NAME,
    'model':      'SimpleGAN',
    'fid':        scores.get('fid'),
    'dacid':      scores.get('dacid'),
    'g_losses':   g_losses,
    'd_losses':   d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Metrics  -> {out}')

---
## Troubleshooting

**`FileNotFoundError: Missing CelebA files`**  
Re-check Step C in the header. The images must be inside `img_align_celeba/`, not in another ZIP.

**`CUDA out of memory`**  
Reduce batch size: add `cfg.BATCH_SIZE = 16` in Cell 3.

**Drive takes too long to mount or images read slowly**  
This is normal with large datasets on Drive. Consider using Colab's local disk (`/content/`) if your session has enough space (~30 GB needed):
```python
# Copy from Drive to local disk (faster reads, but lost on disconnect)
!cp -r /content/drive/MyDrive/celeba_data /content/celeba_data
CELEBA_DRIVE_PATH = '/content/celeba_data'
```